# ICBC Road Safety Analysis — Data Pipeline
**Data source:** ICBC crash data (2020–2024, ~1.37M rows)  

---

| Step | Description |
|------|-------------|
| 0 | Imports & path configuration |
| 1 | Load ICBC data  |
| 2 | Standardize column names |
| 3 | Rename columns |
| 4 | Missingness audit |
| 5 | Clean & type-cast (severity, flags, road user type) |
| 6 | Temporal feature engineering (month, season, hour, cyclic encoding, weekend) |
| 7 | Final quality check |
| 8 | Export to CSV |


## Introduction

This notebook implements the data pipeline for the ICBC stream of our road safety inequality 
project. The analysis uses crash records published by the Insurance Corporation of British 
Columbia (ICBC) covering 2020–2024, representing approximately 1.37 million reported crashes 
across British Columbia. The dataset was selected for its comprehensive provincial coverage — 
since basic ICBC insurance is mandatory in BC, nearly all crashes involving insured vehicles 
are captured, making it one of the most complete crash records available in the province. 
Analysis is limited to this five-year window as ICBC's structured open data format was 
introduced in 2020; prior records exist but follow different reporting conventions shaped 
by legislative changes since 2008, making pre-2020 and post-2020 data difficult to compare 
directly. The 2020 baseline should also be interpreted with caution, as COVID-19 lockdowns 
significantly reduced traffic volumes that year. This pipeline covers data loading, column 
standardization, feature engineering, and export of a cleaned dataset for downstream 
exploratory analysis and modelling. 

## Step 0 — Imports & Path Configuration

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import time
from IPython.display import display, Markdown

# Paths
DATA_DIR   = "data"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ICBC_FILE  = os.path.join(DATA_DIR, "icbc_crashes.csv")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "icbc_cleaned_enriched.csv")

## Step 1 — Load ICBC Data

In [15]:
df = pd.read_csv(
    ICBC_FILE,
    encoding="utf-16",
    sep="\t",
    engine="python",
)
display(Markdown(f"**Shape:** {df.shape[0]:,} rows × {df.shape[1]} columns"))
display(df.dtypes.rename("dtype").to_frame())

**Shape:** 1,368,385 rows × 24 columns

,dtype
Crash Breakdown 2,object
Date Of Loss Year,int64
Animal Flag,object
Crash Severity,object
Cyclist Flag,object
Day Of Week,object
Derived Crash Configuration,object
Heavy Veh Flag,object
Intersection Crash,object
Month Of Year,object


## Step 2 — Standardize Column Names

In [17]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[()\/]", "", regex=True)
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

display(Markdown("**Standardized columns:**"))
display(pd.DataFrame(df.columns, columns=["column_name"]))

**Standardized columns:**

,column_name
0,crash_breakdown_2
1,date_of_loss_year
2,animal_flag
3,crash_severity
4,cyclist_flag
5,day_of_week
6,derived_crash_configuration
7,heavy_veh_flag
8,intersection_crash
9,month_of_year


## Step 3 — Rename Columns

In [19]:
df = df.rename(columns={
    "date_of_loss_year":            "year",
    "month_of_year":                "month",
    "day_of_week":                  "day_of_week",
    "time_category":                "time_category",
    "crash_severity":               "severity",
    "derived_crash_configuration":  "crash_config",
    "road_location_description":    "road_location",
    "municipality_name_ifnull":     "municipality",
    "street_full_name_ifnull":      "street",
    "region":                       "region",
    "intersection_crash":           "intersection",
    "animal_flag":                  "flag_animal",
    "cyclist_flag":                 "flag_cyclist",
    "heavy_veh_flag":               "flag_heavy_veh",
    "motorcycle_flag":              "flag_motorcycle",
    "parked_vehicle_flag":          "flag_parked_vehicle",
    "parking_lot_flag":             "flag_parking_lot",
    "pedestrian_flag":              "flag_pedestrian",
    "total_crashes":                "total_crashes",
    "total_victims":                "total_victims",
    "crash_breakdown_2":            "crash_breakdown",
    "municipality_name":            "municipality_raw",
    "street_full_name":             "street_raw",
    "metric_selector":              "metric_selector",
})

display(Markdown("**Renamed columns:**"))
display(pd.DataFrame(df.columns, columns=["column_name"]))
display(Markdown("**Sample rows:**"))
display(df.head(3))

**Renamed columns:**

,column_name
0,crash_breakdown
1,year
2,flag_animal
3,severity
4,flag_cyclist
5,day_of_week
6,crash_config
7,flag_heavy_veh
8,intersection
9,month


**Sample rows:**

,crash_breakdown,year,flag_animal,severity,flag_cyclist,day_of_week,crash_config,flag_heavy_veh,intersection,month,...,flag_pedestrian,region,road_location,street,time_category,municipality_raw,street_raw,metric_selector,total_crashes,total_victims
0,DELTA,2020,No,CASUALTY CRASH,No,FRIDAY,REAR END,No,No,JANUARY,...,No,LOWER MAINLAND,UNKNOWN,HWY 99,15:00-17:59,DELTA,HWY 99,1,1,1
1,MISSION,2020,No,CASUALTY CRASH,No,MONDAY,CONFLICTED,No,No,JULY,...,No,LOWER MAINLAND,UNKNOWN,MISSION BRIDGE,15:00-17:59,MISSION,MISSION BRIDGE,1,1,3
2,ABBOTSFORD,2020,No,CASUALTY CRASH,No,MONDAY,SINGLE VEHICLE,No,No,DECEMBER,...,No,LOWER MAINLAND,UNKNOWN,VYE RD,18:00-20:59,ABBOTSFORD,VYE RD,1,1,2


## Step 4 — Missingness Audit

In [21]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
report      = pd.DataFrame({"missing_n": missing, "missing_%": missing_pct})
report      = report[report["missing_n"] > 0].sort_values("missing_%", ascending=False)

if report.empty:
    display(Markdown("**No missing values found.**"))
else:
    display(Markdown("**Missing values detected:**"))
    display(report.style.background_gradient(subset=["missing_%"], cmap="Reds"))

**Missing values detected:**

,missing_n,missing_%
street_raw,647,0.050000
street,169,0.010000


## Step 5 — Clean & Type-Cast Columns

### 5a — Severity mapping


In [23]:
# ── Step 5a: Severity mapping ––
severity_map = {"casualty crash": 1, "property damage only": 0}

sev_col = "severity"
df["severity_raw"] = df[sev_col].str.strip().str.lower()
df["severity_ord"] = df["severity_raw"].map(severity_map)
df["is_casualty"]  = (df["severity_ord"] == 1).astype(int)
is_fatal_col = [c for c in df.columns if "fatal" in c]
df["is_fatal"] = df[is_fatal_col[0]] if is_fatal_col else float("nan")

display(Markdown("**Severity distribution:**"))
display(df["severity_raw"].value_counts().rename_axis("severity_raw").reset_index(name="count"))
display(Markdown(f"**is_casualty rate:** {df['is_casualty'].mean()*100:.1f}%"))


**Severity distribution:**

,severity_raw,count
0,property damage only,1124169
1,casualty crash,244216


**is_casualty rate:** 17.8%

### 5b — Binary flags (Yes/No → 1/0)

In [25]:
flag_cols = [c for c in df.columns if c.startswith("flag_")] + ["intersection"]
for col in flag_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0, "Y": 1, "N": 0, 1: 1, 0: 0})

display(Markdown(f"**Flag columns encoded:** `{flag_cols}`"))
display(df[flag_cols].head(5))

**Flag columns encoded:** `['flag_animal', 'flag_cyclist', 'flag_heavy_veh', 'flag_motorcycle', 'flag_parked_vehicle', 'flag_parking_lot', 'flag_pedestrian', 'intersection']`

,flag_animal,flag_cyclist,flag_heavy_veh,flag_motorcycle,flag_parked_vehicle,flag_parking_lot,flag_pedestrian,intersection
0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0


### 5c — Road user type derivation

In [27]:
# Priority: Pedestrian > Cyclist > Motorcyclist > Heavy Vehicle > Car Occupant
def assign_road_user(row):
    if row["flag_pedestrian"] == 1: return "Pedestrian"
    if row["flag_cyclist"]    == 1: return "Cyclist"
    if row["flag_motorcycle"] == 1: return "Motorcyclist"
    if row["flag_heavy_veh"]  == 1: return "Heavy Vehicle"
    return "Car Occupant"

df["road_user_type"] = df.apply(assign_road_user, axis=1)
df["is_vulnerable"]  = (
    (df["flag_pedestrian"] == 1) |
    (df["flag_cyclist"]    == 1) |
    (df["flag_motorcycle"] == 1)
).astype(int)

display(Markdown("**Road user type distribution:**"))
display(
    df["road_user_type"].value_counts()
    .rename_axis("road_user_type").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

multi = df[(df["flag_pedestrian"] + df["flag_cyclist"] + df["flag_motorcycle"]) > 1]
display(Markdown(f"**Crashes with multiple vulnerable flags:** {len(multi):,}"))

**Road user type distribution:**

road_user_type,count,% of total
Car Occupant,1264164,92.380000
Heavy Vehicle,66808,4.880000
Pedestrian,14873,1.090000
Cyclist,12229,0.890000
Motorcyclist,10311,0.750000


**Crashes with multiple vulnerable flags:** 909

## Step 6 — Temporal Feature Engineering

### 6a — Month names → integers & season


In [29]:
month_map = {
    "JANUARY": 1, "FEBRUARY": 2,  "MARCH": 3,    "APRIL": 4,
    "MAY":     5, "JUNE": 6,      "JULY": 7,     "AUGUST": 8,
    "SEPTEMBER": 9, "OCTOBER": 10, "NOVEMBER": 11, "DECEMBER": 12,
}

# Convert to plain Python string first to bypass Arrow dtype
month_str = df["month"].astype(str).str.strip().str.upper()

if month_str.isin(month_map.keys()).any():
    df["month"] = month_str.map(month_map)
else:
    df["month"] = pd.to_numeric(month_str, errors="coerce")

df["month"] = df["month"].astype(float)

bad_months = df[df["month"].isna()]["month"].unique()
if len(bad_months):
    display(Markdown(f" **Unmapped month values:** `{bad_months}`"))
else:
    display(Markdown(" All month values mapped."))

season_map = {
    12: "Winter", 1: "Winter",  2: "Winter",
    3:  "Spring", 4: "Spring",  5: "Spring",
    6:  "Summer", 7: "Summer",  8: "Summer",
    9:  "Fall",  10: "Fall",   11: "Fall",
}
df["season"] = df["month"].map(season_map)

display(Markdown("**Season distribution:**"))
display(
    df["season"].value_counts()
    .rename_axis("season").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

 All month values mapped.

**Season distribution:**

season,count,% of total
Fall,359641,26.280000
Summer,352471,25.760000
Winter,350703,25.630000
Spring,305570,22.330000


### 6b — Time category → approximate hour

In [31]:
time_hour_map = {
    "00:00-02:59":  1, "03:00-05:59":  4, "06:00-08:59":  7,
    "09:00-11:59": 10, "12:00-14:59": 13, "15:00-17:59": 16,
    "18:00-20:59": 19, "21:00-23:59": 22,
}
df["hour_approx"] = df["time_category"].str.strip().map(time_hour_map)

bad_times = df[df["hour_approx"].isna()]["time_category"].unique()
if len(bad_times):
    display(Markdown(f"**Unmapped time categories:** `{bad_times}`"))
else:
    display(Markdown("All time categories mapped."))

display(Markdown("**Time category distribution:**"))
display(df["time_category"].value_counts().rename_axis("time_category").reset_index(name="count"))

All time categories mapped.

**Time category distribution:**

,time_category,count
0,12:00-14:59,346376
1,15:00-17:59,345182
2,09:00-11:59,249230
3,18:00-20:59,168775
4,06:00-08:59,136863
5,21:00-23:59,71702
6,00:00-02:59,25405
7,03:00-05:59,24852


### 6c — Cyclic encoding & weekend flag

In [33]:
# Cyclic encoding — ensures hour 23 and hour 1 are treated as close
df["hour_sin"]  = np.sin(2 * np.pi * df["hour_approx"] / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour_approx"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Day of week & weekend
dow_map = {
    "monday": 0, "tuesday": 1, "wednesday": 2, "thursday": 3,
    "friday": 4, "saturday": 5, "sunday": 6,
}
df["dow_num"]    = df["day_of_week"].str.strip().str.lower().map(dow_map)
df["is_weekend"] = (df["dow_num"] >= 5).astype(int)

display(Markdown("**Sample — temporal features:**"))
display(df[["month", "season", "hour_approx", "hour_sin", "hour_cos", "is_weekend"]].head(5))

**Sample — temporal features:**

,month,season,hour_approx,hour_sin,hour_cos,is_weekend
0,1.0,Winter,16,-0.866025,-0.500000,0
1,7.0,Summer,16,-0.866025,-0.500000,0
2,12.0,Winter,19,-0.965926,0.258819,0
3,9.0,Fall,13,-0.258819,-0.965926,1
4,11.0,Fall,16,-0.866025,-0.500000,1


## Step 8 — Final Quality Check

In [35]:
display(Markdown(f"**Final shape:** {df.shape[0]:,} rows × {df.shape[1]} columns"))

display(Markdown("**Class balance:**"))
display(
    df["severity_raw"].value_counts()
    .rename_axis("severity").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

**Final shape:** 1,368,385 rows × 38 columns

**Class balance:**

severity,count,% of total
property damage only,1124169,82.150000
casualty crash,244216,17.850000


In [36]:
display(Markdown("**Road user type distribution:**"))
display(
    df["road_user_type"].value_counts()
    .rename_axis("road_user_type").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

display(Markdown("**Season distribution:**"))
display(
    df["season"].value_counts()
    .rename_axis("season").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

display(Markdown("**is_casualty by road_user_type:**"))
display(
    df.groupby("road_user_type")["is_casualty"].mean().mul(100).round(1)
    .rename("casualty_rate_%").reset_index()
    .style.hide(axis="index")
)

**Road user type distribution:**

road_user_type,count,% of total
Car Occupant,1264164,92.380000
Heavy Vehicle,66808,4.880000
Pedestrian,14873,1.090000
Cyclist,12229,0.890000
Motorcyclist,10311,0.750000


**Season distribution:**

season,count,% of total
Fall,359641,26.280000
Summer,352471,25.760000
Winter,350703,25.630000
Spring,305570,22.330000


**is_casualty by road_user_type:**

road_user_type,casualty_rate_%
Car Occupant,16.600000
Cyclist,63.700000
Heavy Vehicle,14.100000
Motorcyclist,54.600000
Pedestrian,80.500000


In [37]:
display(Markdown("**Road user type distribution:**"))
display(
    df["road_user_type"].value_counts()
    .rename_axis("road_user_type").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

display(Markdown("**Season distribution:**"))
display(
    df["season"].value_counts()
    .rename_axis("season").reset_index(name="count")
    .assign(**{"% of total": lambda x: (x["count"] / x["count"].sum() * 100).round(2)})
    .style.hide(axis="index")
)

display(Markdown("**is_casualty by road_user_type:**"))
display(
    df.groupby("road_user_type")["is_casualty"].mean().mul(100).round(1)
    .rename("casualty_rate_%").reset_index()
    .style.hide(axis="index")
)

**Road user type distribution:**

road_user_type,count,% of total
Car Occupant,1264164,92.380000
Heavy Vehicle,66808,4.880000
Pedestrian,14873,1.090000
Cyclist,12229,0.890000
Motorcyclist,10311,0.750000


**Season distribution:**

season,count,% of total
Fall,359641,26.280000
Summer,352471,25.760000
Winter,350703,25.630000
Spring,305570,22.330000


**is_casualty by road_user_type:**

road_user_type,casualty_rate_%
Car Occupant,16.600000
Cyclist,63.700000
Heavy Vehicle,14.100000
Motorcyclist,54.600000
Pedestrian,80.500000


## Step 9 — Export

In [39]:
df.to_csv(OUTPUT_CSV, index=False)
display(Markdown(
    f"**Pipeline complete.**\n\n"
    f"`{OUTPUT_CSV}` — {df.shape[0]:,} rows × {df.shape[1]} columns"
))

**Pipeline complete.**

`output/icbc_cleaned_enriched.csv` — 1,368,385 rows × 38 columns